In [1]:
# Fix for protobuf/tensorflow crash in this Kaggle image:
# Force protobuf to use the pure-Python implementation before importing TensorFlow.
import os

os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import tensorflow as tf

print("TF:", tf.__version__)
print("Eager:", tf.executing_eagerly())



2026-05-14 18:48:04.128420: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778784484.141333      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778784484.145298      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF: 2.18.0
Eager: True


In [2]:
ROOT_DIR = "/kaggle/input/"
DATA_DIR = os.path.join(ROOT_DIR, "cassava-leaf-disease-classification")

AUTOTUNE = tf.data.experimental.AUTOTUNE
HEIGHT, WIDTH = 240, 240  # keep original input size
BATCH_SIZE = 32
NUM_CLASSES = 5

TRAIN_TFREC_GLOB = os.path.join(DATA_DIR, "train_tfrecords", "ld_train*.tfrec")
TEST_TFREC_GLOB = os.path.join(DATA_DIR, "test_tfrecords", "ld_test*.tfrec")

SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")
TRAIN_CSV_PATH = os.path.join(DATA_DIR, "train.csv")




In [3]:
def _parse_function(example, feature_description):
    parsed_example = tf.io.parse_single_example(example, feature_description)
    image = tf.io.decode_jpeg(parsed_example["image"], channels=3)
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (HEIGHT, WIDTH))
    if "target" in feature_description:
        target = parsed_example["target"]
        return image, target
    return image, parsed_example["image_name"]


def load_data(path, batch_size=32, train=True):
    filenames = tf.io.gfile.glob(path)
    if len(filenames) == 0:
        raise FileNotFoundError(f"No TFRecord files matched: {path}")

    dataset = tf.data.TFRecordDataset(filenames, num_parallel_reads=AUTOTUNE)
    if train:
        feature_description = {
            "image": tf.io.FixedLenFeature([], tf.string),
            "image_name": tf.io.FixedLenFeature([], tf.string),
            "target": tf.io.FixedLenFeature([], tf.int64),
        }
    else:
        feature_description = {
            "image": tf.io.FixedLenFeature([], tf.string),
            "image_name": tf.io.FixedLenFeature([], tf.string),
        }

    parsed_dataset = dataset.map(
        lambda x: _parse_function(x, feature_description), num_parallel_calls=AUTOTUNE
    )
    return parsed_dataset.batch(batch_size).prefetch(AUTOTUNE)




In [4]:
# Build datasets
full_train_ds = load_data(TRAIN_TFREC_GLOB, batch_size=BATCH_SIZE, train=True)
test_set = load_data(TEST_TFREC_GLOB, batch_size=BATCH_SIZE, train=False)

# Split train/val deterministically without changing core approach (still TFRecord pipeline)
# 18721 train images => 10% val
train_df = pd.read_csv(TRAIN_CSV_PATH)
n_train = len(train_df)
val_count = max(1, int(0.1 * n_train))
train_count = n_train - val_count

train_ds = full_train_ds.take(train_count // BATCH_SIZE)
val_ds = full_train_ds.skip(train_count // BATCH_SIZE)

print("Approx train batches:", train_count // BATCH_SIZE)
print("Val batches:", max(1, val_count // BATCH_SIZE))



Approx train batches: 526
Val batches: 58


I0000 00:00:1778784491.310341      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:c6:00.0, compute capability: 8.6


In [5]:
# Replace missing external .h5 with an in-notebook EfficientNet-based model (same general architecture family).
# This preserves the core idea: EfficientNet -> pooling -> classifier.
base = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(HEIGHT, WIDTH, 3),
)
base.trainable = False  # stable + fast; avoids timeouts

inputs = tf.keras.Input(shape=(HEIGHT, WIDTH, 3))
x = tf.keras.applications.efficientnet.preprocess_input(inputs)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

model.summary()



16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 240, 240, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 8, 8, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,055,976 (15.47 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [6]:
# Train (no early stopping; keep runtime reasonable)
EPOCHS = 3
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=1,
)



Epoch 1/3


I0000 00:00:1778784502.029030     111 service.cc:148] XLA service 0x7f2168002860 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778784502.029069     111 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1778784503.119492     111 cuda_dnn.cc:529] Loaded cuDNN version 90300


      6/Unknown 16s 30ms/step - accuracy: 0.4122 - loss: 1.4367

I0000 00:00:1778784509.111508     111 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


    524/Unknown 26s 18ms/step - accuracy: 0.6652 - loss: 0.9106

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


526/526 ━━━━━━━━━━━━━━━━━━━━ 44s 54ms/step - accuracy: 0.6654 - loss: 0.9099 - val_accuracy: 0.7544 - val_loss: 0.6829
Epoch 2/3
526/526 ━━━━━━━━━━━━━━━━━━━━ 20s 37ms/step - accuracy: 0.7542 - loss: 0.6750 - val_accuracy: 0.7660 - val_loss: 0.6443
Epoch 3/3
526/526 ━━━━━━━━━━━━━━━━━━━━ 23s 42ms/step - accuracy: 0.7644 - loss: 0.6436 - val_accuracy: 0.7734 - val_loss: 0.6302


In [7]:
# Predict on test TFRecords
test_IDs, predictions = [], []
for batch in test_set:
    images, names = batch
    probs = model.predict_on_batch(images)
    pred = np.argmax(probs, axis=1).astype(int)
    # names are bytes tensors
    ID_batch = [n.decode("utf-8") for n in names.numpy().tolist()]
    test_IDs.extend(ID_batch)
    predictions.extend(pred.tolist())

pred_df = pd.DataFrame({"image_id": test_IDs, "label": predictions})

# Ensure submission has exact same length and order as sample_submission (fixes length mismatch).
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
sub = sample_sub[["image_id"]].merge(pred_df, on="image_id", how="left")

# If anything didn't merge, fall back to a valid class (0) to keep submission valid.
sub["label"] = sub["label"].fillna(0).astype(int)

# Final sanity checks
assert sub.shape[0] == sample_sub.shape[0], (sub.shape, sample_sub.shape)
assert list(sub.columns) == ["image_id", "label"]

sub.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", sub.shape)
print(sub.head())

Wrote submission.csv with shape: (2676, 2)
         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      3
3  1234555380.jpg      1
4  1234571117.jpg      3
